# QRAMP — Interactive Demo for Reviewers

**Cryptographic asset inventory and quantum-resistant migration planning**

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.21974850.svg)](https://doi.org/10.5281/zenodo.21974850)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

This notebook runs the **complete QRAMP pipeline** against the shipped testbed. Every cell executes real commands — no pre-computed results.

**Instructions:** Click `Runtime → Run all` or run each cell sequentially with `Shift+Enter`.

Repository: https://github.com/Zhangtijerry/QRAMP

## 0. Install QRAMP

In [ ]:
%%bash
# Clone and install QRAMP with all optional dependencies
git clone --depth 1 https://github.com/Zhangtijerry/QRAMP.git 2>/dev/null || echo 'Already cloned'
cd QRAMP && pip install -q -e '.[source,validate,versions]'
echo '✅ QRAMP installed successfully'
qramp --help | head -5

In [ ]:
import os
os.chdir('/content/QRAMP')
print('Working directory:', os.getcwd())
print('Testbed contents:', os.listdir('testbed/'))

## 1. Collect — Build the cryptographic inventory

Ten pluggable collectors extract evidence from source code, SBOMs, certificates, configurations, and TLS endpoints.

In [ ]:
%%bash
# Authorise the TLS scope (HMAC-signed — the collector refuses unsigned scopes)
qramp sign-scope testbed/testbed.scope.yaml --key testbed/scope.key

# Scan all evidence sources
qramp scan --source testbed/sample-app \
           --sbom   testbed/sample-app/sbom.cdx.json \
           --certs  testbed/certs \
           --config testbed/etc \
           --owners testbed/policy/owners.yaml \
           -o cbom.json

## 2. Assess — Classify and prioritise

The versioned rule pack `transition-2024.3` classifies each asset. The Mosca inequality X+Y−Z computes urgency tiers P0–P3.

In [ ]:
%%bash
qramp assess cbom.json \
      --rules src/qramp/rules/transition-2024.yaml \
      --lifetimes testbed/policy/data-lifetimes.yaml \
      -o assessment.json

## 3. Plan — Generate the migration schedule

Build a prerequisite-ordered, capacity-constrained task DAG.

In [ ]:
%%bash
qramp plan assessment.json \
      --capacity testbed/policy/teams.yaml \
      --horizon 24m \
      -o plan.yaml

## 4. Verify — Machine-checked acceptance predicates

Re-scan and evaluate each task's acceptance predicate. The testbed is **intentionally unremediated**, so some tasks will fail — that is the point.

In [ ]:
%%bash
# Exit code 2 = some tasks failed (expected on unremediated testbed)
qramp verify plan.yaml --against cbom.json --phase 1 \
      --evidence-out evidence.json || true

## 5. Analysis commands

In [ ]:
%%bash
echo '════════════════════════════════════════'
echo '  Migration Readiness'
echo '════════════════════════════════════════'
qramp readiness assessment.json

echo ''
echo '════════════════════════════════════════'
echo '  Upgrade Simulation: OpenSSL → 3.5'
echo '════════════════════════════════════════'
qramp simulate assessment.json --upgrade openssl=3.5

echo ''
echo '════════════════════════════════════════'
echo '  Certificate Chain (weak only)'
echo '════════════════════════════════════════'
qramp chain assessment.json --weak-only

echo ''
echo '════════════════════════════════════════'
echo '  Impact Analysis: OpenSSL'
echo '════════════════════════════════════════'
qramp impact assessment.json --component openssl

## 6. Export — Standards-compliant outputs

In [ ]:
%%bash
# CycloneDX 1.6 CBOM (schema-validated)
qramp export assessment.json --format cyclonedx -o cbom.cdx.json

# SARIF 2.1.0 for GitHub code scanning
qramp export assessment.json --format sarif \
      --rules src/qramp/rules/transition-2024.yaml -o qramp.sarif

# Issues export
qramp export plan.yaml --format issues -o issues.json

echo ''
echo 'Generated files:'
ls -lh cbom.cdx.json qramp.sarif issues.json

## 7. Report — Interactive HTML dashboard

In [ ]:
%%bash
qramp report assessment.json -o dashboard.html
echo "Dashboard size: $(wc -c < dashboard.html) bytes (self-contained, no external dependencies)"

In [ ]:
# Display the dashboard inline
from IPython.display import HTML, display
with open('dashboard.html', 'r') as f:
    html = f.read()
display(HTML(f'<iframe srcdoc="{html.replace(chr(34), "&quot;")}" width="100%" height="700" frameborder="0"></iframe>'))

## 8. Policy tooling — Lint, provenance, diff

In [ ]:
%%bash
echo '── lint-rules ──'
qramp lint-rules src/qramp/rules/transition-2024.yaml --strict

echo ''
echo '── lint-provider-map ──'
qramp lint-provider-map

echo ''
echo '── lint-aliases ──'
qramp lint-aliases

echo ''
echo '── lint-vex ──'
qramp lint-vex testbed/vex/exceptions.yaml

echo ''
echo '── provenance (rules that fired, with standards citations) ──'
qramp provenance --cbom assessment.json --require-references

## 9. Test suite

In [ ]:
%%bash
pip install -q pytest
cd /content/QRAMP && pytest -q 2>&1 | tail -5

---

**All results above are produced by running the real QRAMP tool against the shipped testbed. No pre-computed data is used.**

- Repository: https://github.com/Zhangtijerry/QRAMP
- DOI: [10.5281/zenodo.21974850](https://doi.org/10.5281/zenodo.21974850)
- License: MIT
- User manual: [QRAMP_USER_MANUAL.md](https://github.com/Zhangtijerry/QRAMP/blob/main/QRAMP_USER_MANUAL.md)